# Factor SV (FSV) end-to-end (with robust shocks)\n\nThis notebook demonstrates factor stochastic volatility (full, time-varying covariance)\nand robust shocks (Student-t) on the bundled example dataset.\n\nFeatures demonstrated:\n\n- ELB (shadow-rate augmentation)\n- Factor SV (`covariance="factor"`)\n- Robust shocks (`model.shocks.family="student_t"`)\n- Labeled outputs via `srvar.xarray`\n

In [ ]:
import numpy as np\nimport pandas as pd\n\nfrom srvar import Dataset, ElbSpec, VolatilitySpec\nfrom srvar.api import fit, forecast\nfrom srvar.spec import ModelSpec, PriorSpec, SamplerConfig, ShockSpec\n\ndf = pd.read_csv("../data/example.csv")\ndf["date"] = pd.to_datetime(df["date"])\n\nvalues = df[["r", "y"]].to_numpy(dtype=float)\nds = Dataset.from_arrays(values=values, variables=["r", "y"], time_index=df["date"])\n\nmodel = ModelSpec(\n    p=2,\n    include_intercept=True,\n    elb=ElbSpec(bound=0.0, applies_to=["r"]),\n    volatility=VolatilitySpec(\n        enabled=True,\n        covariance="factor",\n        dynamics="rw",\n        k_factors=1,\n        loading_prior_var=1.0,\n        store_factor_draws=False,\n    ),\n    shocks=ShockSpec(family="student_t", df=7.0),\n)\n\nprior = PriorSpec.niw_default(k=1 + ds.N * model.p, n=ds.N)\nsampler = SamplerConfig(draws=200, burn_in=50, thin=2)\n\nfit_res = fit(ds, model, prior, sampler, rng=np.random.default_rng(0))\nfit_res\n

In [ ]:
fc = forecast(fit_res, horizons=[1, 4, 8], draws=400, rng=np.random.default_rng(1))\nfc.mean\n

## Labeled outputs (`xarray`)\n\nConventions:\n- SV/FSV states are defined on the effective sample `T - p`; the first `p` time points are `NaN`.\n- For factor SV loadings, `ds_fit["loadings"]` is an alias of `ds_fit["lambda"]`.\n

In [ ]:
try:\n    from srvar.xarray import fit_to_xarray, forecast_to_xarray\n\n    ds_fit = fit_to_xarray(fit_res)\n    ds_fc = forecast_to_xarray(fc)\n    print(ds_fit)\n    print(ds_fc)\nexcept ImportError as e:\n    print(e)\n